<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_08_3_keras_hyperparameters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# T81-558: Anwendungen tiefer neuronaler Netzwerke
**Modul 8: Kaggle-Datensätze**
* Dozent: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Jeff Heaton](https://sites.wustl.edu/jeffheaton/)
* Weitere Informationen finden Sie unter [class website](https://sites.wustl.edu/jeffheaton/t81-558/).

# Modul 8 Material

* Teil 8.1: Einführung in Kaggle [[Video]](https://www.youtube.com/watch?v=7Mk46fb0Ayg&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=7Mk46fb0Ayg&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 8.2: Erstellen von Ensembles mit Scikit-Learn und PyTorch [[Video]](https://www.youtube.com/watch?v=przbLRCRL24&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=przbLRCRL24&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* **Teil 8.3: Wie sollten Sie Ihr PyTorch-Neuralnetzwerk strukturieren: Hyperparameter** [[Video]](https://www.youtube.com/watch?v=YTL2BR4U2Ng&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=YTL2BR4U2Ng&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 8.4: Bayesianische Hyperparameteroptimierung für PyTorch [[Video]](https://www.youtube.com/watch?v=1f4psgAcefU&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=1f4psgAcefU&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 8.5: Kaggle des aktuellen Semesters [[Video]] [[Notebook]](t81_558_class_08_5_kaggle_project.ipynb)

# Google CoLab-Anweisungen

Der folgende Code stellt sicher, dass Google CoLab die richtige Version von TensorFlow ausführt.

In [1]:
# Startup CoLab
try:
    import google.colab

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False


# Schön formatierte Zeitzeichenfolge
def hms_string(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int((sec_elapsed % (60 * 60)) / 60)
    s = sec_elapsed % 60
    return "{}:{:>02}:{:>05.2f}".format(h, m, s)

Note: not using Google CoLab


# Teil 8.3: Netzwerkarchitektur: Hyperparameter

Sie haben wahrscheinlich mehrere Hyperparameter bemerkt, die zuvor in diesem Kurs vorgestellt wurden und die Sie für Ihr neuronales Netzwerk auswählen müssen. Die Anzahl der Schichten, die Neuronenanzahl pro Schicht, die Schichttypen und die Aktivierungsfunktionen sind alles Entscheidungen, die Sie treffen müssen, um Ihr neuronales Netzwerk zu optimieren. Einige der Kategorien von Hyperparametern, aus denen Sie auswählen können, stammen aus der folgenden Liste:

* Anzahl der verborgenen Schichten und Neuronenzahlen
* Aktivierungsfunktionen
* Erweiterte Aktivierungsfunktionen
* Regularisierung: L1, L2, Dropout
* Batch-Normalisierung
* Trainingsparameter

In den folgenden Abschnitten werden diese Kategorien für PyTorch vorgestellt. Ich werde zwar einige allgemeine Richtlinien für die Auswahl von Hyperparametern bereitstellen, aber keine zwei Aufgaben sind gleich. Es ist von Vorteil, mit diesen Werten zu experimentieren, um herauszufinden, was für Ihr neuronales Netzwerk am besten funktioniert. Im nächsten Teil werden wir sehen, wie maschinelles Lernen einige dieser Werte unabhängig auswählen kann.

## Anzahl der verborgenen Schichten und Neuronenzahlen

Die Struktur der PyTorch-Ebenen ist vielleicht der Hyperparameter, der den meisten zuerst auffällt. Wie viele Ebenen sollten Sie haben? Wie viele Neuronen befinden sich auf jeder Ebene? Welche Aktivierungsfunktion und welchen Ebenentyp sollten Sie verwenden? Dies sind alles Fragen, die beim Entwerfen eines neuronalen Netzwerks auftauchen. Es gibt viele verschiedene [types of layer](https://pytorch.org/docs/stable/nn.html) in PyTorch, hier aufgelistet:

* **Aktivierung** – PyTorch ermöglicht Ihnen das Hinzufügen von Aktivierungsfunktionen mithilfe von torch.nn-Modulen. Anstelle einer Aktivierungsebene geben Sie die Aktivierungsfunktion normalerweise direkt nach einem linearen (oder anderen) Ebenentyp an.
* **Regularisierung** Für die L1/L2-Regularisierung in PyTorch verwenden Sie im Allgemeinen keine separate Ebene. Stattdessen können Sie beim Einrichten eines Optimierers wie SGD oder Adam eine Gewichtsabnahme hinzufügen. Dies funktioniert als L2-Regularisierung. Für L1 müssen Sie es möglicherweise manuell implementieren.
* **Linear** – Der ursprüngliche Schichttyp des neuronalen Netzwerks. Bei diesem Schichttyp ist jedes Neuron mit der nächsten Schicht verbunden. Der Eingabevektor ist eindimensional und die Platzierung bestimmter Eingaben als nächstes beeinflusst sich nicht gegenseitig.
* **Dropout** - Funktioniert, indem bei jedem Vorwärtsdurchlauf ein Teil der Eingabeeinheiten zufällig auf 0 gesetzt wird, was dabei hilft, Überanpassung zu verhindern. In PyTorch wird Dropout standardmäßig nur während des Trainings angewendet.
* **Abflachen** – Flacht die Eingabe auf 1D ab und hat keinen Einfluss auf die Batchgröße.
* **Permute** – PyTorch-Tensoren verfügen über eine Permute-Methode, mit der die Dimensionen eines Tensors neu angeordnet werden können. Dies ist nützlich, wenn mit verschiedenen Arten von Ebenen gearbeitet wird, die bestimmte Eingabeformen erwarten, und für Aufgaben wie das Verbinden von RNNs und Faltungsnetzwerken.
* **RepeatVector** – Wiederholt die Eingabe n-mal.

Es ist immer ein Versuch und Irrtum, eine gute Anzahl von Neuronen und verborgenen Schichten auszuwählen. Im Allgemeinen ist die Anzahl der Neuronen auf jeder Schicht näher an der verborgenen Schicht größer und kleiner in Richtung der Ausgabeschicht. Diese Konfiguration verleiht dem neuronalen Netzwerk ein eher dreieckiges oder trapezförmiges Aussehen.

## Aktivierungsfunktionen

Aktivierungsfunktionen sind eine Auswahl, die Sie für jede Ebene treffen müssen. Im Allgemeinen können Sie dieser Richtlinie folgen:
* Versteckte Schichten - RELU
* Ausgabeebene – Softmax für die Klassifizierung, linear für die Regression.

Einige der gängigen Aktivierungsfunktionen in PyTorch sind hier aufgeführt:

* **softmax** - Wird für die Klassifizierung mehrerer Klassen verwendet. Stellt sicher, dass sich alle Ausgabeneuronen wie Wahrscheinlichkeiten verhalten und in der Summe 1,0 ergeben.
* **elu** – Exponential Linear Unit. Exponential Linear Unit oder der weithin bekannte Name ELU ist eine Funktion, deren Kosten tendenziell schneller auf Null konvergieren und genauere Ergebnisse liefern. Kann negative Ausgaben erzeugen.
* **selu** – Scaled Exponential Linear Unit (SELU), im Wesentlichen **elu** multipliziert mit einer Skalierungskonstante.
* **softplus** – Softplus-Aktivierungsfunktion. $log(exp(x) + 1)$ [Introduced](https://papers.nips.cc/paper/1920-incorporating-second-order-functional-knowledge-for-better-option-pricing.pdf) im Jahr 2001.
* **Softsign** Softsign-Aktivierungsfunktion. $x / (abs(x) + 1)$ Ähnlich wie tanh, wird aber nicht häufig verwendet.
* **relu** – Sehr beliebte Aktivierungsfunktion für neuronale Netzwerke. Wird für versteckte Schichten verwendet, kann keine negativen Werte ausgeben. Keine trainierbaren Parameter.
* **tanh** Klassische Aktivierungsfunktion für neuronale Netzwerke, die in modernen Netzwerken jedoch häufig durch die Relu-Familie ersetzt wird.
* **Sigmoid** – Klassische Aktivierung eines neuronalen Netzwerks. Wird oft in der Ausgabeebene eines binären Klassifikators verwendet.
* **hard_sigmoid** – rechentechnisch weniger aufwändige Variante von Sigmoid.
* **exp** – Exponentiale (Basis e) Aktivierungsfunktion.

Weitere Informationen zu den Aktivierungsfunktionen von PyTorch finden Sie hier:

* [PyTorch Activation Functions](https://pytorch.org/docs/stable/nn.html)
* [Activation Function Cheat Sheets](https://pytorch.org/docs/stable/nn.html)


## Batch-Normalisierung und Dropout

* [PyTorch Dropout](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html)
* [PyTorch Batch Normalization](https://pytorch.org/docs/stable/generated/torch.nn.functional.batch_norm.html)

* Ioffe, S., & Szegedy, C. (2015). [Batch normalization: Accelerating deep network training by reducing internal covariate shift](https://arxiv.org/abs/1502.03167). *arXiv-Preprint arXiv:1502.03167*.

Normalisieren Sie die Aktivierungen der vorherigen Schicht bei jedem Stapel, d. h. wenden Sie eine Transformation an, die die mittlere Aktivierung nahe 0 und die Aktivierungsstandardabweichung nahe 1 hält. Dies kann eine höhere Lernrate ermöglichen.


## Trainingsparameter

* [PyTorch Optimizers](https://pytorch.org/docs/stable/optim.html)

* **Batchgröße** – Normalerweise klein, etwa 32 oder so.
* **Lernrate** – Normalerweise gering, ungefähr 1e-3.

